# 03 · Concierge router and specialists (lever 2)

**What changed (v1 to v2):** a `claude-haiku-5-5` Concierge (`nestling/concierge.py`, nine routes, its own few-shot examples) routes each question to a specialist (`nestling/agents/`: Care, Nourish, Money, Ready, Celebrate, Journal, plus out-of-scope and clarify replies). Retrieval is still plain dense search.

**Hypothesis (plan §9.5 #2):** routing accuracy +15 to 20 points.

In [1]:
from evals import compare

table = compare.comparison_table()
BEFORE, AFTER = "v1", "v2"
rows = [r for r in table if r["version"] in (BEFORE, AFTER)]
print(compare.format_table(rows))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
v1           100.0% (+0.0%)    99.0% (+1.9%) X    79.8% (-4.8%) X      82.2% (+4.1%)    85.7% (-3.6%) X       7.58 (-0.99)     1.007 (-0.298)     1.513 (+0.011)
v2           100.0% (+0.0%)    99.0% (+0.0%) X    84.6% (+4.8%) X      80.8% (-1.4%)    79.4% (-6.3%) X     8.31 (+0.73) X      0.917 (-0.09)     1.541 (+0.028)
X = misses the §9.1 pass bar (citation match has no bar of its own); deltas are vs the previous row


## Metrics before and after

Saved results only: `evals/results/*_summary.json` plus the Nebius judge CSVs. Deltas are AFTER minus BEFORE; X marks a missed §9.1 pass bar.

In [2]:
for v in ("v1", "v2"):
    print(v, "routing misses:", compare.failure_modes(v)["routing boundary"])

v1 routing misses: ['B08', 'B13', 'B16', 'E06', 'E07', 'F01', 'F02', 'F03', 'F13', 'H16', 'M03', 'M04', 'M06', 'M08', 'M09', 'M11', 'M16', 'M20', 'M21', 'M23', 'M24']
v2 routing misses: ['B08', 'B13', 'E06', 'E07', 'E09', 'F02', 'F03', 'F04', 'F06', 'M03', 'M08', 'M09', 'M10', 'M11', 'M16', 'M21']


## Reading the result

- **Routing improved, far less than predicted:** 79.8% to 84.6% (+4.8 points against a predicted +15 to 20). Misses fell from 21 to 16 cases; the bar is 90%, so it still fails.
- **Regressions, reported plainly:** p95 latency rose 7.58 s to 8.31 s and now misses the 8 s bar (the extra router hop); judged faithfulness fell 85.7% to 79.4%; citation match 82.2% to 80.8%.
- **Cost:** mean cost fell (1.007 to 0.917 INR).
- Remaining routing errors sit on the boundaries (Care vs Nourish, partner prep, postpartum mood). Round-2 Concierge changes in v5 (see notebook 06) address them.

## Live run (off)

Re-running costs money (see the cost column) and writes a new LangSmith experiment. Saved results above are what this notebook reports.

In [3]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
    from nestling import graph

    graph.run_experiment("v2")
else:
    print("RUN_LIVE = False: showing saved results only")

RUN_LIVE = False: showing saved results only
